# ASC 2 — Tạo Pseudo Sentiment Labels (Low Confidence Data)

**Pipeline:**
1. Chạy ASC inference → [neg, neu, pos] probabilities per aspect
2. **Sentence-level splitting:** nếu có ≥1 aspect không vượt ngưỡng → cả câu vào low
3. Concat new high + Phase 1 old high

**Thresholds:** `polar_threshold = 0.9`, `neutral_threshold = 0.55`

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os


BASE_DIR      = "/content/drive/MyDrive/outputs_electronics_cleaning"
SCRIPT_DIR    = BASE_DIR


ASC1_HIGH_DIR = f"{BASE_DIR}/ASC_PHASE_1"
ASC1_LOW_DIR  = f"{BASE_DIR}/ASC_PHASE_1"

ASC2_MODEL_PATH = f"{BASE_DIR}/ASC_PHASE_2/model"

OUTPUT_HIGH_DIR = f"{BASE_DIR}/ASC_2"
OUTPUT_LOW_DIR  = f"{BASE_DIR}/ASC_2"
REPORT_DIR      = f"{BASE_DIR}/ASC_2"

os.makedirs(OUTPUT_HIGH_DIR, exist_ok=True)
os.makedirs(OUTPUT_LOW_DIR,  exist_ok=True)
os.makedirs(REPORT_DIR,      exist_ok=True)


CATEGORY_NAME = "Electronics_part1"

POLAR_THRESHOLD   = 0.9
NEUTRAL_THRESHOLD = 0.55

MAX_LENGTH = 128
CHUNK_SIZE = 100_000

print(f"BASE_DIR        : {BASE_DIR}")
print(f"ASC1_HIGH_DIR   : {ASC1_HIGH_DIR}")
print(f"ASC1_LOW_DIR    : {ASC1_LOW_DIR}")
print(f"ASC2_MODEL_PATH : {ASC2_MODEL_PATH}")
print(f"OUTPUT_HIGH_DIR : {OUTPUT_HIGH_DIR}")
print(f"OUTPUT_LOW_DIR  : {OUTPUT_LOW_DIR}")
print(f"REPORT_DIR      : {REPORT_DIR}")
print(f"CATEGORY_NAME   : {CATEGORY_NAME}")
print(f"polar_threshold : {POLAR_THRESHOLD}  |  neutral_threshold: {NEUTRAL_THRESHOLD}")
print(f"MAX_LENGTH      : {MAX_LENGTH}  |  CHUNK_SIZE: {CHUNK_SIZE:,}")


BASE_DIR        : /content/drive/MyDrive/outputs_electronics_cleaning
ASC1_HIGH_DIR   : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1
ASC1_LOW_DIR    : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1
ASC2_MODEL_PATH : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/model
OUTPUT_HIGH_DIR : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2
OUTPUT_LOW_DIR  : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2
REPORT_DIR      : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2
CATEGORY_NAME   : Electronics_part1
polar_threshold : 0.9  |  neutral_threshold: 0.55
MAX_LENGTH      : 128  |  CHUNK_SIZE: 100,000


In [3]:
import sys
import gc
import queue
import threading

import numpy as np
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pad
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification


if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
from analyze_aspect_sentiment import predict_asc, clean_text, mark_aspect


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {_vram_gb:.1f} GB")
    ASC_BATCH_SIZE = 1024 if _vram_gb >= 40 else 512 if _vram_gb >= 15 else 256

    # CUDA performance flags
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    ASC_BATCH_SIZE = 32
    print("WARNING: GPU không khả dụng, tốc độ sẽ rất chậm")

# Load Phase 2 retrained model — thay thế Phase 1 teacher trong module
import analyze_aspect_sentiment as _asc_mod

_asc_mod.asc_tokenizer = AutoTokenizer.from_pretrained(ASC2_MODEL_PATH, use_fast=True)
_asc_mod.asc_model = AutoModelForSequenceClassification.from_pretrained(ASC2_MODEL_PATH).to(device)
_asc_mod.asc_model.eval()
print(f"Phase 2 model loaded: {ASC2_MODEL_PATH}")
print(f"Labels: {_asc_mod.asc_model.config.id2label}")


def predict_aspect_sentiment(df, polar_threshold, neutral_threshold, batch_size, max_length):
    """
    Sentence-level ASC với strict splitting rule:
      HIGH → TẤT CẢ aspects của câu đều vượt ngưỡng confidence
      LOW  → ít nhất 1 aspect không vượt ngưỡng → toàn bộ câu vào low
    """
    if len(df) == 0:
        return df.assign(sentiments=pd.NA, gate_confidence=pd.NA), df.copy()

    exp = (
        df["aspects"].explode()
        .reset_index()
        .rename(columns={"index": "_orig_idx", "aspects": "_aspect"})
    )
    exp["_sentence_text"] = df["sentence_text"].to_numpy()[exp["_orig_idx"].to_numpy()]

    results = predict_asc(
        sentences=exp["_sentence_text"].tolist(),
        aspects=exp["_aspect"].tolist(),
        polar_threshold=polar_threshold,
        neutral_threshold=neutral_threshold,
        batch_size=batch_size,
        max_length=max_length,
    )

    exp["_conf"]    = [r[0] for r in results]
    exp["_is_high"] = [r[1] for r in results]

    agg = (
        exp.groupby("_orig_idx", sort=False)
        .agg(_all_high=("_is_high", "all"), sentiments=("_conf", list))
    )

    out = df.join(agg, how="left")
    high_mask = out["_all_high"].fillna(False).astype(bool)

    high_df = out[high_mask].drop(columns=["_all_high"]).copy()
    low_df  = out[~high_mask].drop(columns=["_all_high", "sentiments"], errors="ignore").copy()

    if len(high_df) > 0:
        high_df["gate_confidence"] = high_df["sentiments"].apply(
            lambda sents: float(min(max(s) for s in sents))
        )

    return high_df, low_df


if device == "cuda":
    _asc_mod.asc_model = _asc_mod.asc_model.half()
    print("asc_model → FP16  ✓")

    ASC_BATCH_SIZE = 2048 if _vram_gb >= 40 else 1024 if _vram_gb >= 15 else 512
    print(f"ASC batch size (FP16)  : {ASC_BATCH_SIZE}")

    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    if _tv >= (2, 0):
        try:
            _asc_mod.asc_model = torch.compile(_asc_mod.asc_model, mode="default")
            print("asc_model → torch.compile  ✓  (batch đầu sẽ chậm hơn do compile)")
        except Exception as _ce:
            print(f"torch.compile không khả dụng: {_ce}")

print(f"\nASC batch size : {ASC_BATCH_SIZE}")
print(f"MAX_LENGTH     : {MAX_LENGTH}")
print(f"CHUNK_SIZE     : {CHUNK_SIZE:,}")
print("predict_aspect_sentiment (sentence-level) defined OK")
print("Imports OK")


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Phase 2 model loaded: /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/model
Labels: {0: 'negative', 1: 'neutral', 2: 'positive'}
asc_model → FP16  ✓
ASC batch size (FP16)  : 1024
asc_model → torch.compile  ✓  (batch đầu sẽ chậm hơn do compile)

ASC batch size : 1024
MAX_LENGTH     : 128
CHUNK_SIZE     : 100,000
predict_aspect_sentiment (sentence-level) defined OK
Imports OK


In [4]:

LOW_INPUT_PATH  = f"{ASC1_LOW_DIR}/low_confidence_{CATEGORY_NAME}.parquet"
HIGH_INPUT_PATH = f"{ASC1_HIGH_DIR}/high_confidence_{CATEGORY_NAME}.parquet"

assert os.path.exists(LOW_INPUT_PATH),  f"Không tìm thấy: {LOW_INPUT_PATH}"
assert os.path.exists(HIGH_INPUT_PATH), f"Không tìm thấy: {HIGH_INPUT_PATH}"

# Đếm rows
ds_low  = pad.dataset(LOW_INPUT_PATH,  format="parquet")
ds_high = pad.dataset(HIGH_INPUT_PATH, format="parquet")
n_low_input   = ds_low.count_rows()
n_phase1_high = ds_high.count_rows()

print(f"low_confidence input  : {n_low_input:>12,} rows  ({LOW_INPUT_PATH})")
print(f"high_confidence Phase1: {n_phase1_high:>12,} rows  ({HIGH_INPUT_PATH})")


_sample = pd.read_parquet(LOW_INPUT_PATH, columns=None).head(5)
print(f"\nSchema low input:")
print(_sample.dtypes)
print(f"\nSample (5 dòng đầu):")
print(_sample)
print(f"\nPhân phối số aspect/câu:")
print(_sample["aspects"].apply(len).value_counts().sort_index())
del _sample

low_confidence input  :    2,998,256 rows  (/content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/low_confidence_Electronics_part1.parquet)
high_confidence Phase1:   12,990,920 rows  (/content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/high_confidence_Electronics_part1.parquet)

Schema low input:
parent_asin       object
sentence_id        int32
sentence_text     object
rating           float64
category_name     object
aspects           object
dtype: object

Sample (5 dòng đầu):
  parent_asin  sentence_id                                      sentence_text  \
0  B00BCSYZSW            2  nice look antenna sometimes wonky slim profile...   
1  B01LVVG5XS            3  obviously bass is lacking but the speakers don...   
2  B007HCCOD0            4  br my kindle has no back light but that has be...   
3  B07DRKQRRS            4  because of the signal strength at 52 degrees i...   
4  B07R82333S            1             movies might be okay text was terrible   

   rati

In [5]:
def _iter_chunks(parquet_path, chunk_size, prefetch=3):
    """Stream parquet file theo chunks với prefetch trên CPU thread riêng."""
    ds = pad.dataset(parquet_path, format="parquet")
    q  = queue.Queue(maxsize=prefetch)

    def _producer():
        try:
            for batch in ds.scanner(batch_size=chunk_size).to_batches():
                q.put(batch.to_pandas())
                del batch
        finally:
            q.put(None)

    threading.Thread(target=_producer, daemon=True).start()
    while True:
        item = q.get()
        if item is None:
            break
        yield item


HIGH_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "category_name", "gate_confidence",
    "aspects", "sentiments",
]
LOW_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "category_name", "aspects",
]

# Output hỗ trợ resume
TEMP_HIGH_PATH = f"{OUTPUT_HIGH_DIR}/_temp_new_high.parquet"
TEMP_LOW_PATH  = f"{OUTPUT_LOW_DIR}/_temp_new_low.parquet"

# Resume nếu đã xử lý xong
if os.path.exists(TEMP_HIGH_PATH) and os.path.exists(TEMP_LOW_PATH):
    total_new_high = pad.dataset(TEMP_HIGH_PATH, format="parquet").count_rows()
    total_new_low  = pad.dataset(TEMP_LOW_PATH,  format="parquet").count_rows()
    print(f"[SKIP] Temp files đã tồn tại — new_high={total_new_high:,}  new_low={total_new_low:,}")
    print("Xóa temp files nếu muốn chạy lại từ đầu.")
else:
    n_chunks    = (n_low_input + CHUNK_SIZE - 1) // CHUNK_SIZE
    writer_high = writer_low = None
    total_new_high = total_new_low = 0

    print(f"Bắt đầu inference trên {n_low_input:,} rows ({n_chunks} chunks)...")
    try:
        for _ci, chunk in enumerate(
            tqdm(_iter_chunks(LOW_INPUT_PATH, CHUNK_SIZE), total=n_chunks, desc="ASC inference"),
            1,
        ):
            # Chỉ xử lý câu có ít nhất 1 aspect
            chunk = chunk[chunk["aspects"].apply(len) >= 1].reset_index(drop=True)
            if len(chunk) == 0:
                continue

            high_df, low_df = predict_aspect_sentiment(
                df=chunk,
                polar_threshold=POLAR_THRESHOLD,
                neutral_threshold=NEUTRAL_THRESHOLD,
                batch_size=ASC_BATCH_SIZE,
                max_length=MAX_LENGTH,
            )
            del chunk

            if len(high_df) > 0:
                high_table = pa.Table.from_pandas(high_df[HIGH_COLS], preserve_index=False)
                if writer_high is None:
                    writer_high = pq.ParquetWriter(TEMP_HIGH_PATH, high_table.schema)
                writer_high.write_table(high_table)
                del high_table

            if len(low_df) > 0:
                low_table = pa.Table.from_pandas(low_df[LOW_COLS], preserve_index=False)
                if writer_low is None:
                    writer_low = pq.ParquetWriter(TEMP_LOW_PATH, low_table.schema)
                writer_low.write_table(low_table)
                del low_table

            total_new_high += len(high_df)
            total_new_low  += len(low_df)
            del high_df, low_df

            # Cleanup mỗi 10 chunk thay vì mỗi chunk — tránh GPU sync tốn kém
            if _ci % 10 == 0:
                gc.collect()
                if device == "cuda":
                    torch.cuda.empty_cache()

    finally:
        if writer_high: writer_high.close()
        if writer_low:  writer_low.close()
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    print(f"\nInference hoàn tất.")
    print(f"  new_high : {total_new_high:>12,}  ({total_new_high/n_low_input*100:.2f}%)")
    print(f"  new_low  : {total_new_low:>12,}  ({total_new_low/n_low_input*100:.2f}%)")


Bắt đầu inference trên 2,998,256 rows (30 chunks)...


ASC inference:   0%|          | 0/30 [00:00<?, ?it/s]

W0521 14:05:39.884000 1440 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode



Inference hoàn tất.
  new_high :    2,811,090  (93.76%)
  new_low  :      187,166  (6.24%)


In [6]:

print("=== Phase 2 Inference Summary ===")
print(f"  Phase 1 low input : {n_low_input:>12,}")
print(f"  New high          : {total_new_high:>12,}  ({total_new_high/n_low_input*100:.2f}%)")
print(f"  New low           : {total_new_low:>12,}  ({total_new_low/n_low_input*100:.2f}%)")

if os.path.exists(TEMP_HIGH_PATH):
    _chk = pd.read_parquet(TEMP_HIGH_PATH).head(3)
    print(f"\nSample new_high (3 dòng đầu):")
    print(_chk[["sentence_text", "aspects", "sentiments"]].to_string())

    _sents = _chk["sentiments"].iloc[0]
    # Parquet / PyArrow có thể deserialize list-of-list thành numpy array — accept cả hai
    assert isinstance(_sents, (list, np.ndarray)), \
        f"sentiments phải là list hoặc array, got {type(_sents)}"
    assert isinstance(_sents[0], (list, tuple, np.ndarray)), \
        f"mỗi phần tử sentiments phải là list/tuple/array [neg, neu, pos], got {type(_sents[0])}"
    assert len(_sents[0]) == 3, \
        f"mỗi triplet phải có 3 phần tử, got {len(_sents[0])}"
    print("\nsentiments format OK: list of [neg, neu, pos] triplets ✓")
    del _chk

print("Sentence-level splitting rule: câu vào HIGH chỉ khi ALL aspects vượt ngưỡng ✓")

=== Phase 2 Inference Summary ===
  Phase 1 low input :    2,998,256
  New high          :    2,811,090  (93.76%)
  New low           :      187,166  (6.24%)

Sample new_high (3 dòng đầu):
                                                                                                           sentence_text           aspects                                                                                                                                  sentiments
0                                                       nice look antenna sometimes wonky slim profile under the cabinet   [profile, look]  [[0.9999773502349854, 9.83739664661698e-06, 1.2780362339981366e-05], [1.6799331206129864e-05, 1.4063808521314058e-05, 0.9999691247940063]]
1  obviously bass is lacking but the speakers don t suffer from than tinny sound that cheaper speakers often suffer from  [speakers, bass]  [[0.0009198746993206441, 0.00011832750169560313, 0.9989618062973022], [0.9999762773513794, 1.5720097508165054e-05

In [7]:

OUT_HIGH_PATH = f"{OUTPUT_HIGH_DIR}/high_confidence_samples_{CATEGORY_NAME}.parquet"

print(f"Phase 1 high rows : {n_phase1_high:>12,}")
print(f"Phase 2 new high  : {total_new_high:>12,}")
print(f"Tổng dự kiến      : {n_phase1_high + total_new_high:>12,}")

writer_combined = None
n_written = 0

for src_path in tqdm([HIGH_INPUT_PATH, TEMP_HIGH_PATH], desc="Concat high"):
    if not os.path.exists(src_path):
        print(f"[SKIP] Không tìm thấy: {src_path}")
        continue
    tbl = pq.read_table(src_path, columns=HIGH_COLS)
    if writer_combined is None:
        writer_combined = pq.ParquetWriter(OUT_HIGH_PATH, tbl.schema)
    writer_combined.write_table(tbl)
    n_written += len(tbl)
    del tbl

if writer_combined:
    writer_combined.close()

print(f"\nĐã ghi : {OUT_HIGH_PATH}")
print(f"Tổng rows : {n_written:,}")

# Kiểm tra duplicate sentence_id
_verify = pq.read_table(OUT_HIGH_PATH, columns=["sentence_id"]).to_pandas()
n_dup = _verify["sentence_id"].duplicated().sum()
print(f"Duplicate sentence_id : {n_dup:,}" + (" ← cần kiểm tra!" if n_dup > 0 else " ✓"))
del _verify

Phase 1 high rows :   12,990,920
Phase 2 new high  :    2,811,090
Tổng dự kiến      :   15,802,010


Concat high:   0%|          | 0/2 [00:00<?, ?it/s]


Đã ghi : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2/high_confidence_samples_Electronics_part1.parquet
Tổng rows : 15,802,010
Duplicate sentence_id : 15,801,780 ← cần kiểm tra!


In [8]:

OUT_LOW_PATH = f"{OUTPUT_LOW_DIR}/low_confidence_samples_{CATEGORY_NAME}.parquet"

os.replace(TEMP_LOW_PATH, OUT_LOW_PATH)
n_low_written = pad.dataset(OUT_LOW_PATH, format="parquet").count_rows()

print(f"Đã ghi : {OUT_LOW_PATH}")
print(f"Tổng low rows : {n_low_written:,}")

print("\n--- Schema high_confidence_samples ---")
for field in pq.read_schema(OUT_HIGH_PATH):
    print(f"  {field.name:25s} {field.type}")

print("\n--- Schema low_confidence_samples ---")
for field in pq.read_schema(OUT_LOW_PATH):
    print(f"  {field.name:25s} {field.type}")

Đã ghi : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2/low_confidence_samples_Electronics_part1.parquet
Tổng low rows : 187,166

--- Schema high_confidence_samples ---
  parent_asin               string
  sentence_id               int32
  sentence_text             string
  rating                    double
  category_name             string
  gate_confidence           double
  aspects                   list<element: string>
  sentiments                list<element: list<element: double>>

--- Schema low_confidence_samples ---
  parent_asin               string
  sentence_id               int32
  sentence_text             string
  rating                    double
  category_name             string
  aspects                   list<element: string>


In [9]:

n_high_combined = pq.read_metadata(OUT_HIGH_PATH).num_rows
n_low_final     = n_low_written

n_pos = n_neu = n_neg = total_asp = 0
pf    = pq.ParquetFile(OUT_HIGH_PATH)
n_batches = -(-n_high_combined // 200_000)

for batch in tqdm(
    pf.iter_batches(batch_size=200_000, columns=["sentiments"]),
    total=n_batches,
    desc="Computing sentiment stats",
):
    df_b = batch.to_pandas()

    all_trips = np.vstack(
        [np.asarray(row, dtype=np.float32) for row in df_b["sentiments"].explode()]
    )
    doms      = np.argmax(all_trips, axis=1)   # 0=neg, 1=neu, 2=pos
    n_neg    += int((doms == 0).sum())
    n_neu    += int((doms == 1).sum())
    n_pos    += int((doms == 2).sum())
    total_asp += len(doms)
    del df_b, all_trips, doms


def pct(x, total):
    return f"{x / total * 100:.2f}%" if total > 0 else "N/A"


print("=" * 62)
print("  ASC Phase 2 — Pseudo Labeling Report")
print("=" * 62)
print(f"\n[Input]")
print(f"  Phase 1 low confidence  : {n_low_input:>12,}")
print(f"\n[Phase 2 Inference Output]")
print(f"  New high confidence     : {total_new_high:>12,}  ({pct(total_new_high, n_low_input)})")
print(f"  New low  confidence     : {total_new_low:>12,}  ({pct(total_new_low,  n_low_input)})")
print(f"\n[Combined High Confidence (Phase 1 + Phase 2)]")
print(f"  Phase 1 high            : {n_phase1_high:>12,}")
print(f"  Phase 2 new high        : {total_new_high:>12,}")
print(f"  Combined total          : {n_high_combined:>12,}")
print(f"\n[Thresholds]")
print(f"  polar_threshold   : {POLAR_THRESHOLD}")
print(f"  neutral_threshold : {NEUTRAL_THRESHOLD}")
print(f"\n[Sentiment Distribution — Combined High Confidence]")
print(f"  Total aspects     : {total_asp:>12,}")
print(f"  Positive          : {n_pos:>12,}  ({pct(n_pos, total_asp)})")
print(f"  Neutral           : {n_neu:>12,}  ({pct(n_neu, total_asp)})")
print(f"  Negative          : {n_neg:>12,}  ({pct(n_neg, total_asp)})")
print(f"\n[Output files]")
print(f"  high: {OUT_HIGH_PATH}")
print(f"  low : {OUT_LOW_PATH}")
print("=" * 62)

Computing sentiment stats:   0%|          | 0/80 [00:00<?, ?it/s]

  ASC Phase 2 — Pseudo Labeling Report

[Input]
  Phase 1 low confidence  :    2,998,256

[Phase 2 Inference Output]
  New high confidence     :    2,811,090  (93.76%)
  New low  confidence     :      187,166  (6.24%)

[Combined High Confidence (Phase 1 + Phase 2)]
  Phase 1 high            :   12,990,920
  Phase 2 new high        :    2,811,090
  Combined total          :   15,802,010

[Thresholds]
  polar_threshold   : 0.9
  neutral_threshold : 0.55

[Sentiment Distribution — Combined High Confidence]
  Total aspects     :   20,270,822
  Positive          :   13,322,918  (65.72%)
  Neutral           :      175,519  (0.87%)
  Negative          :    6,772,385  (33.41%)

[Output files]
  high: /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2/high_confidence_samples_Electronics_part1.parquet
  low : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2/low_confidence_samples_Electronics_part1.parquet


In [11]:

report_path = f"{REPORT_DIR}/asc_2_report_{CATEGORY_NAME}.txt"

report_lines = [
    "=" * 62,
    "  ASC Phase 2 — Pseudo Labeling Report",
    f"  Category : {CATEGORY_NAME}",
    "=" * 62,
    "",
    "[Input — ASC Phase 1 Low Confidence]",
    f"  Phase 1 low conf total : {n_low_input:>12,}",
    "",
    "[Phase 2 Inference Output]",
    f"  New high confidence  : {total_new_high:>12,}  ({pct(total_new_high, n_low_input)})",
    f"  New low  confidence  : {total_new_low:>12,}  ({pct(total_new_low,  n_low_input)})",
    "",
    "[Combined High Confidence (Phase 1 + Phase 2)]",
    f"  Phase 1 high     : {n_phase1_high:>12,}",
    f"  Phase 2 new high : {total_new_high:>12,}",
    f"  Combined total   : {n_high_combined:>12,}",
    "",

    "[Sentiment Distribution — Combined High Confidence]",
    f"  Total aspects classified : {total_asp:>12,}",
    f"  Positive : {n_pos:>10,}  ({pct(n_pos, total_asp)})",
    f"  Neutral  : {n_neu:>10,}  ({pct(n_neu, total_asp)})",
    f"  Negative : {n_neg:>10,}  ({pct(n_neg, total_asp)})",
    "",
    "[Output Files]",
    f"  high : {OUT_HIGH_PATH}",
    f"  low  : {OUT_LOW_PATH}",
    "",
    "=" * 62,
]

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Báo cáo đã lưu: {report_path}")

# Cleanup temp files
import shutil
_cleanup = input("Xóa temp files? (y/n): ").strip().lower()
if _cleanup == "y":
    for p in [TEMP_HIGH_PATH, TEMP_LOW_PATH]:
        if os.path.exists(p):
            os.remove(p)
    print("Temp files đã xóa.")
else:
    print(f"Temp files giữ lại: {TEMP_HIGH_PATH}")
    print(f"                    {TEMP_LOW_PATH}")

Báo cáo đã lưu: /content/drive/MyDrive/outputs_electronics_cleaning/ASC_2/asc_2_report_Electronics_part1.txt
Xóa temp files? (y/n): y
Temp files đã xóa.
